# 02 – Huấn luyện CTGAN (AI1) và sinh dữ liệu (AI3)
Chạy sau notebook 01. Luồng: chuẩn bị dữ liệu → huấn luyện CTGAN → sinh dữ liệu theo quota từng lớp → kiểm tra nhanh.

**Lần đầu hãy để `SMOKE_TEST = True`** (chạy vài phút để chắc code chạy). Sau đó đặt `False` để huấn luyện thật.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import config, data

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

DATASET = "unsw_nb15"   # đổi thành "cicids2017" để chạy dataset kia

from src.ctgan_pipeline import train_ctgan, sample_by_class, quick_fidelity

SMOKE_TEST = True
EPOCHS = 5 if SMOKE_TEST else 300
CAP_PER_CLASS = 500 if SMOKE_TEST else 3000   # giới hạn số dòng mỗi lớp lớn (lớp hiếm giữ nguyên)


## 1. Chuẩn bị dữ liệu cho CTGAN

In [ ]:
train, test = data.load(DATASET)
real, LABEL, DISCRETE = data.prepare_for_ctgan(train, DATASET, cap_per_class=CAP_PER_CLASS)
print("Bảng huấn luyện:", real.shape)
print("Cột rời rạc:", DISCRETE)
display(data.class_counts(real, LABEL))

## 2. Huấn luyện CTGAN

In [ ]:
import time

# True = nạp mô hình đã lưu trong outputs/ (bỏ qua huấn luyện, ví dụ khi kernel đã tắt)
LOAD_SAVED_MODEL = False
model_path = config.OUT_DIR / f"ctgan_{DATASET}.pt"

if LOAD_SAVED_MODEL:
    from ctgan import CTGAN
    model = CTGAN.load(str(model_path))
    print("Đã nạp mô hình từ", model_path)
else:
    t0 = time.time()
    model = train_ctgan(real, DISCRETE, epochs=EPOCHS, batch_size=500, pac=10, cuda=False)
    print(f"Huấn luyện xong sau {time.time()-t0:.0f} giây")
    config.OUT_DIR.mkdir(exist_ok=True)
    model.save(str(model_path))

## 3. Sinh dữ liệu theo quota từng lớp (AI3)
Sinh số bản ghi **cố định cho mỗi lớp**, không theo tỷ lệ gốc, để lớp hiếm không bị lấn át.

CTGAN chỉ dùng điều kiện lớp như một gợi ý, nhãn sinh ra có thể khác lớp được yêu cầu. Vì vậy hàm `sample_by_class`
sinh dư và chỉ giữ dòng có nhãn đúng lớp, rồi in bảng `match_rate` (tỷ lệ khớp nhãn của từng lớp).
Lớp có `match_rate` rất thấp là lớp CTGAN khó sinh đúng: đây là kết quả nghiên cứu, không phải lỗi mã.

In [ ]:
classes = real[LABEL].unique()
QUOTA = 200 if SMOKE_TEST else 2000
quotas = {c: QUOTA for c in classes}
synth = sample_by_class(model, LABEL, quotas)
print(synth.shape)
synth.to_csv(config.OUT_DIR / f"synthetic_ctgan_{DATASET}.csv", index=False)
synth.head(20)

## 4. Kiểm tra nhanh chất lượng
Khoảng cách giữa dữ liệu thật và dữ liệu sinh theo từng cột (càng gần 0 càng giống). Đây chỉ là kiểm tra sơ bộ; phần fidelity chính thức thuộc nhóm Data Science (DS3).

In [ ]:
fid = quick_fidelity(real, synth, DISCRETE)
print("Khoảng cách trung bình:", round(fid["distance"].mean(), 3))
fid.head(15)

In [ ]:
cmp = pd.DataFrame({"thật": real[LABEL].value_counts(), "sinh": synth[LABEL].value_counts()}).fillna(0).astype(int)
cmp

## Bước tiếp theo
- `03_train_tvae.ipynb` – TVAE (AI-2) cùng quy trình, để so sánh với CTGAN.
- `04_quality_check.ipynb` – kiểm tra hợp lệ, trùng lặp, khoảng cách tới bản ghi huấn luyện (AI-3).
- `05_ids_baseline.ipynb` – Random Forest trên dữ liệu thật và thật + dữ liệu sinh (AI-4).
- AI-5: lọc / sinh lại theo điểm rủi ro MIA của nhóm Cyber Security (chờ điểm rủi ro)